# Pipeline de Dados em Nuvem — ETL com base real

Este notebook implementa um fluxo ETL (*Extract, Transform, Load*) usando Python e Pandas, agora aplicado à base pública **Online Retail**, disponibilizada pelo UCI Machine Learning Repository.

**Atenção:** o processamento é executado no ambiente do Colab/Jupyter e os resultados são salvos em arquivos CSV e JSON. Nesta versão, isso ainda não representa uma implantação em serviço de nuvem.

## Fonte dos dados
Chen, D. (2015). *Online Retail*. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

A base reúne 541.909 linhas de transações de uma empresa varejista on-line do Reino Unido, referentes ao período de 1º de dezembro de 2010 a 9 de dezembro de 2011. A licença informada pelo repositório é CC BY 4.0.

## Regras de tratamento
- Padronizar nomes de colunas e tipos;
- Remover linhas integralmente duplicadas;
- Desconsiderar faturas de cancelamento (identificadas pelo prefixo `C`);
- Descartar registros sem data, código/nome de produto ou quantidade/preço válidos;
- Descartar quantidades e preços não positivos;
- Calcular o valor bruto da linha (`quantidade × preço unitário`);
- Preservar país e identificador de cliente quando disponíveis.

O identificador da fatura é mantido como texto, pois pode conter letras nos registros de cancelamento.


## 1. Preparação do ambiente

In [ ]:
# Se estiver no Google Colab/Jupyter e ainda não tiver a biblioteca:
%pip -q install ucimlrepo openpyxl

from pathlib import Path
import json
import pandas as pd
from ucimlrepo import fetch_ucirepo

BASE_DIR = Path('/content/pipeline-dados-nuvem')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR = BASE_DIR / 'outputs'

for folder in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Pastas preparadas em:', BASE_DIR)


## 2. Extração — obtenção da base real

In [ ]:
# Busca a base diretamente no UCI Machine Learning Repository (dataset id 352).
online_retail = fetch_ucirepo(id=352)

# O conjunto original possui variáveis de transação e metadados.
df = online_retail.data.original.copy()

print('Dimensão da base original:', df.shape)
print('Colunas:', df.columns.tolist())
display(df.head())


## 3. Transformação e validação

In [ ]:
rows_input = len(df)

# Padronização dos nomes das colunas
df.columns = (
    df.columns.astype(str).str.strip().str.lower()
      .str.replace(' ', '_', regex=False)
)

required = ['invoiceno', 'stockcode', 'description',
            'quantity', 'invoicedate', 'unitprice']
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f'Colunas obrigatórias ausentes na fonte: {missing_columns}')

# Conversão de tipos
df['invoiceno'] = df['invoiceno'].astype('string').str.strip()
df['stockcode'] = df['stockcode'].astype('string').str.strip()
df['description'] = df['description'].astype('string').str.strip()
df['invoicedate'] = pd.to_datetime(df['invoicedate'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unitprice'] = pd.to_numeric(df['unitprice'], errors='coerce')

# Remoção de duplicatas exatas
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

# Cancelamentos: número da fatura iniciado por C
cancellation_mask = df['invoiceno'].str.upper().str.startswith('C', na=False)
cancellations_removed = int(cancellation_mask.sum())
df = df.loc[~cancellation_mask].copy()

# Regras de validade para as linhas de venda
valid_mask = (
    df['invoiceno'].notna() & df['invoiceno'].ne('') &
    df['stockcode'].notna() & df['stockcode'].ne('') &
    df['description'].notna() & df['description'].ne('') &
    df['invoicedate'].notna() &
    df['quantity'].notna() & df['quantity'].gt(0) &
    df['unitprice'].notna() & df['unitprice'].gt(0)
)

invalid_rows = int((~valid_mask).sum())
clean = df.loc[valid_mask].copy()

# Esquema de saída organizado para análise de vendas
rename_map = {
    'invoiceno': 'id_venda',
    'stockcode': 'codigo_produto',
    'description': 'produto',
    'invoicedate': 'data',
    'quantity': 'quantidade',
    'unitprice': 'valor_unitario',
    'customerid': 'id_cliente',
    'country': 'pais'
}
clean = clean.rename(columns=rename_map)
clean['valor_total'] = (clean['quantidade'] * clean['valor_unitario']).round(2)
clean['data'] = clean['data'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Seleciona as colunas disponíveis, preservando cliente e país se existirem
output_columns = [
    'id_venda', 'data', 'codigo_produto', 'produto', 'quantidade',
    'valor_unitario', 'valor_total', 'id_cliente', 'pais'
]
output_columns = [c for c in output_columns if c in clean.columns]
clean = clean[output_columns].reset_index(drop=True)

print('Registros após tratamento:', len(clean))
display(clean.head(10))


## 4. Carga — gravação dos dados e relatório de qualidade

In [ ]:
output_file = PROCESSED_DIR / 'online_retail_tratado.csv'
report_file = OUTPUT_DIR / 'relatorio_qualidade.json'

clean.to_csv(output_file, index=False, encoding='utf-8-sig')

report = {
    'status': 'sucesso',
    'fonte': 'UCI Machine Learning Repository — Online Retail (id 352)',
    'registros_entrada': int(rows_input),
    'duplicatas_removidas': int(duplicates_removed),
    'cancelamentos_removidos': int(cancellations_removed),
    'registros_invalidos_descartados': int(invalid_rows),
    'registros_saida': int(len(clean)),
    'percentual_retido_sobre_entrada': round(len(clean) / rows_input * 100, 2) if rows_input else 0.0,
    'periodo_data_minima': str(clean['data'].min()) if len(clean) else None,
    'periodo_data_maxima': str(clean['data'].max()) if len(clean) else None,
    'valor_total_vendas': round(float(clean['valor_total'].sum()), 2) if len(clean) else 0.0,
    'arquivo_saida': str(output_file),
    'colunas_saida': list(clean.columns)
}

with open(report_file, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))
print('CSV tratado:', output_file)
print('Relatório JSON:', report_file)


## 5. Indicadores descritivos

In [ ]:
if len(clean):
    print('Quantidade de linhas tratadas:', len(clean))
    print('Faturas distintas:', clean['id_venda'].nunique())
    print('Produtos distintos:', clean['codigo_produto'].nunique())
    if 'pais' in clean.columns:
        print('Países distintos:', clean['pais'].nunique())
    print('Valor bruto total (GBP):', round(clean['valor_total'].sum(), 2))
    display(clean.groupby('pais', dropna=False)['valor_total']
                 .sum().sort_values(ascending=False).head(10).reset_index()
                 if 'pais' in clean.columns else clean.head())
else:
    print('Nenhum registro válido após as regras de tratamento.')


## 6. Download dos resultados (Google Colab)

In [ ]:
try:
    from google.colab import files
    files.download(str(output_file))
    files.download(str(report_file))
except ImportError:
    print('Fora do Google Colab. Os arquivos estão salvos em:', PROCESSED_DIR, 'e', OUTPUT_DIR)


## 7. Interpretação e limitações

O relatório JSON registra os volumes de entrada e saída, duplicatas, cancelamentos, registros inválidos, percentual retido e indicadores de vendas. Os valores finais dependem da execução do notebook e devem ser utilizados na seção de resultados do trabalho.

O pipeline é executado em memória e salva arquivos localmente no ambiente de execução. Portanto, apesar do nome do projeto, esta versão não implementa armazenamento em nuvem, agendamento, monitoramento contínuo ou banco de dados gerenciado. Essas funcionalidades permanecem como evoluções futuras.
